# Reco & True Cluster Variable Distributions -- AFTER Beam Window Cut

Simple per-cluster VARIABLE DISTRIBUTIONS for the charge-light matching
pipeline, drawn on exactly the population
`Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb` evaluates: the same
input tree, the same cuts, the same beam-window cut on the reco side, the same
1-to-1 true-reco pairing. This notebook computes no completeness or purity plots
-- it runs the pipeline only as far as it must to know *which* clusters are
selected and which reco cluster each true neutrino was matched to, then
histograms their variables.

**Variables**

| side | variables |
|---|---|
| reco | average X, average Y, average Z of the cluster (cm); the cluster's bridged optical flash time (us); the cluster's reco charge (ADC, coarse bins) |
| true | average X, average Y, average Z of the cluster (cm); the cluster's true energy (MeV, 100 MeV bins) |

There is no true-side flash time: true clusters carry no flash and no time
(`build_true_points_charge_light` fills the time column with zeros), so timing
stays a reco-side quantity -- the same reasoning as
`metadata.build_true_cluster_type_records`.

**Two populations**, each holding the full set tree below:

| directory | population |
|---|---|
| `all_true_all_selected_reco_clusters/` | every selected reco cluster, every true cluster -- the two sides are different objects and their counts need not agree |
| `pair_true_reco_clusters/` | only the 1-to-1 matched pairs whose true side is a neutrino -- the same objects seen twice, so a difference between the sides is reconstruction rather than population |

In the pairs version the reco set definitions still apply on top, so
`01_all_selected_clusters` and `02_matched_to_true_neutrino` necessarily hold
the same clusters there.

**Contents of each population directory** -- one selection per side, so any two
directories being compared describe the same clusters:

| directory | contents |
|---|---|
| `reco/` | that population's reco clusters |
| `true/` | that population's true **neutrino** clusters, in and out of volume together |
| `reco_true_comparison/` | the two overlaid in the same bins |

Each also carries a `*_cluster_variables.txt` listing the clusters behind its
histograms.

**Binning and style** (all in `draw_variables.py`): the spatial variables use
fixed **50 cm** bins (`SPATIAL_BIN_WIDTH_CM`) over the volume bounds, coarse
enough that a run with few events still has shape rather than one cluster per
bin. True energy uses **100 MeV** bins (`ENERGY_BIN_WIDTH_MEV`) and reco charge
**2e6 ADC** bins (`CHARGE_BIN_WIDTH_ADC`); those two have no fixed detector
range, so the width is fixed and the range follows the data — a high-energy
outlier gets its own bin instead of falling off the end of a fixed axis.
Reco is drawn as a red dotted step line, true as black with sqrt(N)
statistical uncertainties (a line on its own, points in the comparison, where a
second line would just be overwritten where the two agree).

An empty selection still writes its text table stating `clusters: 0`, so an
empty directory always means the code did not run rather than the selection
being genuinely empty.

**Neutrino identification is by cluster ID, not by a type string**:
`reassign_cluster_ID_true_charge_light` gives interaction `nu_idx` the true
cluster id `99990+nu_idx`, so `true_cluster_id >= 99990` is exactly "neutrino"
and `true_cluster_id - 99990` is its `nu_idx` -- an exact key, no spatial
matching and no tolerance to tune.

All plotting lives in `draw_variables.py` (this directory), which is additive:
it changes nothing in the existing evaluation modules and only consumes objects
they already build. Output goes to
`AnalysisDistributions_Reco_True/multi_file_plots_charge_light_matching/RecoTrue_Distributions_AfterTimeWindowCut/`,
so it never touches the evaluation notebooks' plot trees.


In [1]:
# Run scope -- same knobs as Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb.
# Charge-light matching is a combined-APA evaluation (img-global / sed-sce are
# already global across APAs) -- no per-APA/face looping.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None   # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

# Range of event numbers to run, inclusive on both ends: (1, 5) runs events
# 1,2,3,4,5. None runs every event. Applied on top of target_event, so leave
# target_event = None when using a range.
target_event_range = None   # e.g. (1, 5) for events 1..5

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. The `files = N` knob above still takes the
# first N in lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9

# Fail fast rather than silently processing nothing: `evt != target_event` can
# never be False for a tuple, so target_event = (1, 5) would skip every event.
if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")


# ========================================================================
# Decide which levels to draw
# ========================================================================
# Every level draws the SAME six sets; they differ only in how many events are
# pooled into each histogram. Event-level histograms hold a handful of clusters
# each -- useful for debugging one event, noise for physics -- so turn them off
# for a full-statistics run.
b_draw_event_level_plots = False   # one set of distributions per event
b_draw_file_level_plots  = False   # one set of distributions per file
b_draw_job_level_plots   = True   # one set of distributions for the whole job


In [2]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in AnalysisDistributions_Reco_True/, one level below the
# repository root where the pipeline modules and the input trees are. Resolve
# both explicitly so the notebook runs whether Jupyter was started in this
# directory (the usual case) or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "AnalysisDistributions_Reco_True":
    NB_DIR = NB_DIR / "AnalysisDistributions_Reco_True"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")


Job started at: 2026-08-04 17:39:40
Notebook directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/AnalysisDistributions_Reco_True
Repository root:    /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction


In [3]:
# Pipeline modules (repository root) -- imported and used UNCHANGED. This
# notebook runs the evaluation pipeline only as far as the 1-to-1 true-reco
# pairing, which is what set 2 onwards needs; nothing past that (completeness /
# purity plotting, heatmaps, match multiplicity) is drawn here.
from readfiles import ensure_data_extracted, read_charge_light_files_for_event, flatten_mc_tree
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light, reassign_cluster_ID_reco,
    apply_energy_cutoff, apply_min_true_points_cutoff, apply_min_reco_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
)
from cluster_category import cluster_category
from completeness_purity_estimate import EvaluateCompleteness, EvaluatePurity
from clusterpairmatching import MatchTrueToReco1to1
from metadata import (
    build_cluster_flash_metadata, build_img_cluster_flash_metadata,
    add_metadata_true_reco_pair_cluster,
    build_true_cluster_type_records, build_neutrino_vertex_records,
)
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US

# The new module for this notebook (AnalysisDistributions_Reco_True/draw_variables.py):
# record builders, the selectors that define the five reco sets and the true set,
# and the histogram drawer.
from draw_variables import (
    build_reco_flash_time_lookup, build_reco_cluster_variable_records,
    build_true_cluster_variable_records, draw_all_reco_true_variable_sets_versions,
    select_true_neutrino_records, select_matched_pair_records, VERSION_DIRNAME,
)


In [4]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-img-global.json                       (reco clusters, imaging level)
#   file0/data/0/0-clustering-global.json                (reco clusters, post charge-light matching)
#   file0/data/0/0-sed-smear_readout.json                (true clusters)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/0/0-op.json                               (optical/light info)
#   file0/data/1/, 2/, ... (one subdirectory per event)

# The DEAD-AREA-PREPROCESSED tree, produced once by preprocess_deadarea_cut.py.
# Its true-point files already have the dead-area cut applied, which is why
# Apply_deadarea_cut is False below. Read that script's docstring, or
# DEADAREA_PREPROCESSING.txt inside the tree, before switching this back to the
# raw tree: the two are NOT interchangeable.
PARENT_DIR = REPO_ROOT / "Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut"

# Number of files/events to process (convert the 'files'/'events' knobs above)
num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# Output directory: inside AnalysisDistributions_Reco_True, so this notebook's
# output tree is self-contained and never shares a directory with the
# evaluation notebooks' plots.
PLOTBASEDIR = NB_DIR / "multi_file_plots_charge_light_matching" / "RecoTrue_Distributions_AfterTimeWindowCut"
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS -- identical to
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb, deliberately: these
# distributions are meant to describe the population that notebook evaluates,
# so any change here breaks that correspondence.
# ========================================================================
radius_completeness         = 2
radius_purity_xz          = 2
radius_purity_yz          = 5
radius_purity_xy          = 5
min_recopoints_threshold  = 5

# min_true_points_cutoff / min_reco_points_cutoff are DISABLED: this format's
# point clouds are much sparser than the old imaging-based reconstruction --
# real neutrino clusters have been seen with as few as 13 points -- so the old
# threshold (200) would delete real signal clusters outright.
#
# min_cluster_energy IS applied: sed-smear's per-point 'e' field (MeV) is a
# genuine energy deposit, so the old threshold (100 MeV) carries over directly.
min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)
min_reco_points_cutoff    = 200     # NOT APPLIED (Apply_min_reco_points_cutoff = False below)

Apply_energy_cutoff                         = True
Apply_min_true_points_cutoff                = False
Apply_min_reco_points_cutoff                = False
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# BEAM-WINDOW (time) CUT: keeps only reco clusters whose bridged flash time lies
# inside [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US] = [0.33, 1.93] us, i.e. the
# in-spill, neutrino-dominated population. RECO-side only by design -- "in beam
# window" is not a truth quantity, so the true side keeps its cosmic clusters.
Apply_beam_window_cut                       = True
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Set this True only if you point
# PARENT_DIR back at a raw tree.
Apply_deadarea_cut                          = False

# Wire-readout sensitive volume (detector geometry, unit: cm). Also the bounds
# for the vertex_in_volume flag that sets 05 and the true set select on, and the
# axis ranges draw_variables uses for the avg X/Y/Z histograms.
x_min = -250.0
x_max = 250.0
y_min = -200.0
y_max = 200.0
z_min = 0.15
z_max = 500.85

# Metadata label only -- there is no 2-view/3-view distinction in the
# charge-light format, so this is just a constant.
view = "combined"

print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied (threshold {min_cluster_energy} MeV, using sed-smear's per-point 'e' field)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_beam_window_cut:
    print(f"- Beam window cut applied to RECO clusters only ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us flash time)")
else:
    print(f"- Beam window cut NOT applied")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR)")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# ensure_data_extracted() only unzips if that file's data/ folder doesn't
# already exist, so re-running this notebook never re-extracts.
if PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


Configuration:
Parent directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
Plot base directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/AnalysisDistributions_Reco_True/multi_file_plots_charge_light_matching/RecoTrue_Distributions_AfterTimeWindowCut
Files to process: all
Events to process: all

Cuts applied:
- Energy cutoff applied (threshold 100 MeV, using sed-smear's per-point 'e' field)
- Wire readout sensitive xz plane cut applied
- Beam window cut applied to RECO clusters only (0.33 - 1.93 us flash time)
- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR)


In [5]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs


def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                events.append(int(item.name))
            except ValueError:
                pass

    return sorted(events)


# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


Scanning parent directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
------------------------------------------------------------
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file1
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file10
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file11
Found: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file2

In [6]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA reco/true variable distributions
# ============================================================================
# The selection chain below is a copy of
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb's, truncated after the
# 1-to-1 pairing: true side = sed-smear_readout grouped by REAL_CLUSTER_ID and
# reassigned to 99990+nu_idx (neutrino, one cluster per interaction) / avg-X
# (cosmic); reco side = clustering-global grouped by REAL_CLUSTER_ID (NOT
# cluster_id, which can merge physically distinct tracks), beam-window cut,
# fiducial cut, then relabelled by avg-X via reassign_cluster_ID_reco.
#
# THE ONE PIECE THAT IS NEW HERE is the flash-time lookup. reassign_cluster_ID_reco
# relabels reco clusters by avg-X and so destroys clustering-global's
# real_cluster_id -- the namespace the flash records are keyed in. So
# build_reco_flash_time_lookup() is called on predicted_points BEFORE that
# relabelling and reproduces the same round(mean(x), 3) key, giving
# {reassigned_reco_cluster_id: flash_time} for the clusters the pipeline goes on
# to use. Calling it after the relabelling would silently match nothing.

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

job_reco_var_records     = []   # one record per selected reco cluster (build_reco_cluster_variable_records)
job_true_var_records     = []   # one record per selected true cluster (build_true_cluster_variable_records)
job_pair_metadata_list   = []   # per 1-to-1 true-reco pair (add_metadata_true_reco_pair_cluster)
job_cluster_type_records = []   # per-true-cluster is_neutrino (build_true_cluster_type_records)
job_vertex_records       = []   # per true neutrino interaction (build_neutrino_vertex_records)
total_events_processed   = 0
total_files_processed    = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    # SELECTIVE FILTERING: Skip files outside target_file_range (inclusive both
    # ends, matched on the directory name's trailing index -- see
    # file_index_from_name). Applied on top of target_file, not instead of it.
    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    file_output_dir = output_dir / input_file_name
    file_output_dir.mkdir(parents=True, exist_ok=True)

    file_reco_var_records     = []
    file_true_var_records     = []
    file_pair_metadata_list   = []
    file_cluster_type_records = []
    file_vertex_records       = []

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1

    # Start of event loop
    for evt in range(event_low, event_high):
        # SELECTIVE FILTERING: Skip events that don't match target_event
        if target_event is not None and evt != target_event:
            continue

        # SELECTIVE FILTERING: Skip events outside target_event_range (inclusive
        # both ends). Applied on top of target_event, not instead of it.
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key        = f"{input_file_name}_{evt}"
        event_output_dir = file_output_dir / f"event_{evt:03d}"
        event_output_dir.mkdir(parents=True, exist_ok=True)

        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result['true_clustering']
        x_clu,  y_clu,  z_clu,  id_clu,  q_clu,  real_id_clu                       = result['clustering']
        mc_tree = result['mc']
        op_data = result['op']

        # ------------------------------------------------------------------
        # FLASH RECORDS: op.json flashes attached to img-global clusters, then
        # bridged onto clustering-global clusters by point charge ('q').
        # These give both the beam-window cut's cluster list and, further down,
        # every selected reco cluster's flash_time variable.
        # ------------------------------------------------------------------
        event_flash_metadata_list = build_cluster_flash_metadata(
            op_data, input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)

        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}

        # ------------------------------------------------------------------
        # TRUE POINTS: sed-smear_readout in the standard 7-column shape
        # (energy = per-point 'e' in MeV; q_true = 'nu_idx', 0=cosmic,
        # 1/2/...=which neutrino interaction), reassigned to 99990+nu_idx
        # (neutrino) / avg-X (cosmic), then cut.
        # real_id_true (real_cluster_id), NOT id_true: cluster_id is a coarser
        # grouping that can merge physically distinct tracks.
        # ------------------------------------------------------------------
        true_points = build_true_points_charge_light(
            x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points = reassign_cluster_ID_true_charge_light(true_points)

        # Snapshot BEFORE the cuts: build_neutrino_vertex_records uses it to say
        # what a removed neutrino actually deposited. Grouping only, no filtering.
        clusters_true_precut = GroupClustersByID(true_points)

        if Apply_energy_cutoff:
            true_points = apply_energy_cutoff(true_points, min_cluster_energy)
        if Apply_min_true_points_cutoff:
            true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            true_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_points, x_min, x_max, y_min, y_max, z_min, z_max)
        if Apply_deadarea_cut:
            true_points = apply_deadarea_cut_true_charge_light(true_points, output_dir=event_output_dir, event=evt, file_name=input_file_name)

        if len(true_points) == 0:
            print(f"  Event {evt}: no true points remain after cuts, skipping")
            continue

        clusters_true = GroupClustersByID(true_points)

        # ------------------------------------------------------------------
        # RECO POINTS: clustering-global (post charge-light matching), grouped
        # by REAL_CLUSTER_ID. Beam-window cut FIRST -- clu_beam_window_ids lives
        # in the real_cluster_id namespace, which reassign_cluster_ID_reco
        # destroys, so filtering after it would match nothing.
        # ------------------------------------------------------------------
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))

        if Apply_beam_window_cut:
            n_reco_points_before_beam   = len(predicted_points)
            n_reco_clusters_before_beam = len(np.unique(predicted_points[:, 3])) if n_reco_points_before_beam else 0
            beam_ids_array   = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
            print(f"  Event {evt}: beam-window cut kept "
                  f"{len(clu_beam_window_ids)}/{n_reco_clusters_before_beam} reco clusters, "
                  f"{len(predicted_points)}/{n_reco_points_before_beam} reco points")

        if Apply_min_reco_points_cutoff:
            predicted_points = apply_min_reco_points_cutoff(predicted_points, min_reco_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            predicted_points = apply_wire_readout_sensitive_yz_plane_cut_reco(predicted_points, x_min, x_max, y_min, y_max, z_min, z_max)

        # An event can legitimately end up with NO in-spill reco cluster. It is
        # kept rather than skipped -- its true clusters are genuine
        # reconstruction failures and still belong in the true distributions --
        # but reassign_cluster_ID_reco cannot take an empty array, so
        # short-circuit to an empty dict.
        if len(predicted_points) == 0:
            clusters_reco, flash_time_by_reco_id = {}, {}
            print(f"  Event {evt}: no reco cluster survives the beam-window cut")
        else:
            # BEFORE reassign_cluster_ID_reco -- see the header note.
            flash_time_by_reco_id = build_reco_flash_time_lookup(predicted_points, event_img_cluster_flash_records)
            predicted_points      = reassign_cluster_ID_reco(predicted_points)
            clusters_reco         = GroupClustersByID(predicted_points)

        # ------------------------------------------------------------------
        # 1-TO-1 TRUE-RECO PAIRING (existing functions, unchanged). Completeness
        # and purity are computed only because MatchTrueToReco1to1 needs them:
        # sets 2-5 are defined as "the reco cluster the evaluation called this
        # true neutrino's best match", so the pairing must be the identical one
        # the evaluation notebook produces, not a re-derivation.
        # ------------------------------------------------------------------
        cluster_category_results = cluster_category(clusters_true, output_dir=None, event=evt, apa="Combined", file_name=input_file_name)
        completeness_results       = EvaluateCompleteness(clusters_true, clusters_reco, event_key, radius_completeness, min_recopoints_threshold)
        purity_results           = EvaluatePurity(clusters_true, clusters_reco, event_key, radius_purity_xz, radius_purity_yz, radius_purity_xy)

        event_matched_pairs      = MatchTrueToReco1to1(completeness_results, purity_results)
        event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
            event_matched_pairs, cluster_category_results,
            file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)

        # Neutrino/cosmic label records (set 3 counts neutrinos per event from
        # these) and true neutrino interaction vertices from mc.json, joined to
        # their true cluster by nu_idx (cluster_id = 99990+nu_idx, an exact key).
        # vertex_in_volume uses the same bounds as the fiducial cut, and is what
        # set 5 and the true set select on.
        event_cluster_type_records = build_true_cluster_type_records(
            clusters_true, input_file_name, evt, event_key)
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(mc_tree), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        # ------------------------------------------------------------------
        # VARIABLE RECORDS + DRAWING (draw_variables.py)
        # ------------------------------------------------------------------
        event_reco_var_records = build_reco_cluster_variable_records(
            clusters_reco, input_file_name, evt, event_key, "Combined",
            flash_time_by_reco_id=flash_time_by_reco_id)
        event_true_var_records = build_true_cluster_variable_records(
            clusters_true, input_file_name, evt, event_key, "Combined",
            vertex_records=event_vertex_records)

        if b_draw_event_level_plots:
            draw_all_reco_true_variable_sets_versions(
                event_reco_var_records, event_true_var_records, event_pair_metadata_list,
                event_output_dir, f"Event {evt}", f"event_{evt}", "Combined",
                file_name=input_file_name)
            plt.close('all')

        # ------------------------------------------------------------------
        # AGGREGATE TO FILE AND JOB LEVEL
        # ------------------------------------------------------------------
        file_reco_var_records.extend(event_reco_var_records)
        file_true_var_records.extend(event_true_var_records)
        file_pair_metadata_list.extend(event_pair_metadata_list)
        file_cluster_type_records.extend(event_cluster_type_records)
        file_vertex_records.extend(event_vertex_records)

        job_reco_var_records.extend(event_reco_var_records)
        job_true_var_records.extend(event_true_var_records)
        job_pair_metadata_list.extend(event_pair_metadata_list)
        job_cluster_type_records.extend(event_cluster_type_records)
        job_vertex_records.extend(event_vertex_records)

        n_with_flash    = sum(1 for r in event_reco_var_records if r['flash_time'] is not None)
        n_true_neutrino = sum(1 for r in event_true_var_records if r['is_neutrino'])
        print(
            f"  Event {evt}: "
            f"reco clusters={len(event_reco_var_records)} (with flash time={n_with_flash}), "
            f"true clusters={len(event_true_var_records)} (neutrino={n_true_neutrino}), "
            f"1-to-1 pairs={len(event_pair_metadata_list)}, "
            f"neutrino interactions in mc={len(event_vertex_records)}"
        )
        total_events_processed += 1

    # ========================================================================
    # FILE-LEVEL DISTRIBUTIONS: the same six sets over every event in this file
    # ========================================================================
    if b_draw_file_level_plots and (file_reco_var_records or file_true_var_records):
        print(f"\n  FILE-LEVEL AGGREGATION ({input_file_name}): "
              f"{len(file_reco_var_records)} reco clusters, {len(file_true_var_records)} true clusters, "
              f"{len(file_pair_metadata_list)} 1-to-1 pairs")
        draw_all_reco_true_variable_sets_versions(
            file_reco_var_records, file_true_var_records, file_pair_metadata_list,
            file_output_dir / "file_summary", "File Level", "file", "Combined",
            file_name=input_file_name)
        plt.close('all')

# ============================================================================
# JOB-LEVEL DISTRIBUTIONS: the same six sets over every file and event
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total selected reco clusters: {len(job_reco_var_records)}")
print(f"Total selected true clusters: {len(job_true_var_records)}")
print(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
print(f"{'='*70}")

job_output_dir = output_dir / "job_summary"
job_output_dir.mkdir(parents=True, exist_ok=True)

if b_draw_job_level_plots:
    draw_all_reco_true_variable_sets_versions(
        job_reco_var_records, job_true_var_records, job_pair_metadata_list,
        job_output_dir, "Job Level", "job", "Combined")
    plt.close('all')

# ============================================================================
# JOB SUMMARY TEXT FILE -- configuration, and how many clusters ended up in
# each set (recomputed with the same selectors draw_all_reco_true_variable_sets
# uses, so these counts are the histogram entry counts).
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime   = time.time() - job_start_time

job_paired_reco, job_paired_true = select_matched_pair_records(
    job_reco_var_records, job_true_var_records, job_pair_metadata_list)
job_paired_reco, job_paired_true = select_matched_pair_records(
    job_reco_var_records, job_true_var_records, job_pair_metadata_list)
job_true_neutrinos = select_true_neutrino_records(job_true_var_records)

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY -- RECO/TRUE CLUSTER VARIABLE DISTRIBUTIONS")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        summary_lines.append(f"Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("Cuts (identical to Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb):")
summary_lines.append(f"  energy cutoff:            {Apply_energy_cutoff} ({min_cluster_energy} MeV)")
summary_lines.append(f"  min true points cutoff:   {Apply_min_true_points_cutoff} ({min_true_points_cutoff})")
summary_lines.append(f"  min reco points cutoff:   {Apply_min_reco_points_cutoff} ({min_reco_points_cutoff})")
summary_lines.append(f"  wire readout volume cut:  {Apply_wire_readout_sensitive_xz_plane_cut}")
summary_lines.append(f"  beam window cut (reco):   {Apply_beam_window_cut} ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us)")
summary_lines.append(f"  dead area cut here:       {Apply_deadarea_cut} (applied upstream when False)")
summary_lines.append(f"  volume bounds: x [{x_min}, {x_max}], y [{y_min}, {y_max}], z [{z_min}, {z_max}] cm")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
summary_lines.append(f"Total selected reco clusters: {len(job_reco_var_records)}")
summary_lines.append(f"Total selected true clusters: {len(job_true_var_records)}")
summary_lines.append(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
summary_lines.append(f"Total true neutrino interactions (mc.json): {len(job_vertex_records)}")
summary_lines.append("")
summary_lines.append("Populations, each drawn as reco/ + true/ + reco_true_comparison/")
summary_lines.append("(= histogram entries):")
summary_lines.append(f"  {VERSION_DIRNAME['all']}/")
summary_lines.append(f"    reco: {len(job_reco_var_records)} selected clusters")
summary_lines.append(f"    true: {len(job_true_neutrinos)} neutrino clusters "
                     f"(of {len(job_true_var_records)} true clusters)")
summary_lines.append(f"  {VERSION_DIRNAME['pairs']}/")
summary_lines.append(f"    reco: {len(job_paired_reco)} clusters of 1-to-1 pairs")
summary_lines.append(f"    true: {len(job_paired_true)} neutrino clusters of 1-to-1 pairs")
summary_lines.append("")

n_missing_flash = sum(1 for r in job_reco_var_records if r['flash_time'] is None)
summary_lines.append(f"Reco clusters with no flash time (absent from the flash_time histogram): {n_missing_flash}")
if job_reco_var_records and n_missing_flash < len(job_reco_var_records):
    all_flash_times = [r['flash_time'] for r in job_reco_var_records if r['flash_time'] is not None]
    summary_lines.append(f"  mean flash time:   {np.mean(all_flash_times):.4f} us")
    summary_lines.append(f"  median flash time: {np.median(all_flash_times):.4f} us")
    summary_lines.append(f"  min flash time:    {np.min(all_flash_times):.4f} us")
    summary_lines.append(f"  max flash time:    {np.max(all_flash_times):.4f} us")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append("=" * 80)

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")



Output directory: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/AnalysisDistributions_Reco_True/multi_file_plots_charge_light_matching/RecoTrue_Distributions_AfterTimeWindowCut/combined_apa_20260804_173941


FILE 1/12: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Processing events 0 to 9

  Event 0: beam-window cut kept 0/15 reco clusters, 0/21848 reco points
  Event 0: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 0: reco clusters=0 (with flash time=0), true clusters=7 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=1
  Event 1: beam-window cut kept 1/13 reco clusters, 740/18105 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 1: reco clusters=1 (with flash time=1), true clusters=6 (neutrino=1), 1-to-1 pairs=1, neutrino interactions in mc=1
  Event 2: beam-window cu

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 110: beam-window cut kept 1/17 reco clusters, 29/19670 reco points

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 110: reco clusters=1 (with flash time=1), true clusters=7 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=4


/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 111: beam-window cut kept 1/13 reco clusters, 3050/23839 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 111: reco clusters=1 (with flash time=1), true clusters=7 (neutrino=1), 1-to-1 pairs=1, neutrino interactions in mc=2
  Event 112: beam-window cut kept 0/9 reco clusters, 0/36943 reco points
  Event 112: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 112: reco clusters=0 (with flash time=0), true clusters=7 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=1
  Event 113: beam-window cut kept 0/18 reco clusters, 0/27063 reco points
  Event 113: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 113: reco clusters=0 (with flash time=0), true clusters=9 (neutrino=1), 1-to-1 pairs=0, neutrino interactions in mc=3
  Event 114: beam-window cut kept 1/15 reco clusters, 1976/30596 reco points

Found 1 matched pairs of true 

...

  Event 67: beam-window cut kept 1/13 reco clusters, 1864/22983 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 67: reco clusters=1 (with flash time=1), true clusters=10 (neutrino=1), 1-to-1 pairs=1, neutrino interactions in mc=1
  Event 68: beam-window cut kept 0/18 reco clusters, 0/33795 reco points
  Event 68: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 68: reco clusters=0 (with flash time=0), true clusters=10 (neutrino=1), 1-to-1 pairs=0, neutrino interactions in mc=2
  Event 69: beam-window cut kept 0/26 reco clusters, 0/39852 reco points
  Event 69: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 69: reco clusters=0 (with flash time=0), true clusters=15 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=3

FILE 10/12: /Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/Haiwang_files_charge_light_matching_M

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 83: reco clusters=1 (with flash time=1), true clusters=8 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=1
  Event 84: beam-window cut kept 0/12 reco clusters, 0/36107 reco points
  Event 84: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 84: reco clusters=0 (with flash time=0), true clusters=9 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=1
  Event 85: beam-window cut kept 3/19 reco clusters, 5030/27999 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 85: reco clusters=3 (with flash time=3), true clusters=9 (neutrino=1), 1-to-1 pairs=1, neutrino interactions in mc=2
  Event 86: beam-window cut kept 0/22 reco clusters, 0/24661 reco points
  Event 86: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 86: reco clusters=0 (with flash time=0), true clusters=11 (neutrino=0), 1-to-1 pairs=0, neutrino interactio

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(completeness_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 92: reco clusters=1 (with flash time=1), true clusters=8 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=3
  Event 93: beam-window cut kept 0/17 reco clusters, 0/51567 reco points
  Event 93: no reco cluster survives the beam-window cut

Found 0 matched pairs of true and reco clusters (1-to-1)
  Event 93: reco clusters=0 (with flash time=0), true clusters=11 (neutrino=0), 1-to-1 pairs=0, neutrino interactions in mc=2
  Event 94: beam-window cut kept 1/9 reco clusters, 544/25110 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 94: reco clusters=1 (with flash time=1), true clusters=8 (neutrino=1), 1-to-1 pairs=1, neutrino interactions in mc=1
  Event 95: beam-window cut kept 1/12 reco clusters, 1560/9127 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
  Event 95: reco clusters=1 (with flash time=1), true clusters=7 (neutrino=1), 1-to-1 pairs=1, neutrino interactions in mc=1
  Event 96: beam-window cut kept 1/22 reco clu